In [ ]:
import os 
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
from solve_kis import (
    solve_kis,
    multi_query_search,
    reciprocal_rank_fusion,
    fusion_clip_caption
)

from clip_encoder import ClipEncoder
from search_clip import ClipRetriever
from caption_retriever import CaptionRetriever

In [ ]:
encoder = ClipEncoder()
retriever = ClipRetriever()

print("CLIP OK")

In [ ]:
caption_retriever = CaptionRetriever()

print("Caption Retriever OK")

In [ ]:
import json

with open("test.json", "r", encoding="utf-8") as f:
    structured_query = json.load(f)

print(json.dumps(structured_query, ensure_ascii=False, indent=2))

In [ ]:
query_variants = (
    structured_query.get("query_variants")
    or [structured_query.get("raw_query", "")]
)

print(
    f"\n[CLIP Search] "
    f"{len(query_variants)} query variants"
)

clip_multi_results = multi_query_search(
    query_variants,
    encoder,
    retriever,
    top_k=100
)

clip_results = reciprocal_rank_fusion(
    clip_multi_results
)

print("\n=== TOP 20 CLIP ===")

for i, r in enumerate(clip_results[:20], 1):
    print(
        f"{i:02d}. "
        f"video={r['video_id']} "
        f"frame={r['frame_id']} "
        f"score={r['score']:.6f}"
    )

In [ ]:
caption_multi_results = []

print(
    f"\n[Caption Search] "
    f"{len(query_variants)} query variants"
)

for query in query_variants:

    print(f"  Query: {query}")

    results = caption_retriever.search(
        query,
        top_k=100
    )

    caption_multi_results.append(results)


caption_results = reciprocal_rank_fusion(
    caption_multi_results
)

print("\n=== TOP 20 CAPTION ===")

for i, r in enumerate(
    caption_results[:20],
    1
):
    print(
        f"{i:02d}. "
        f"video={r['video_id']} "
        f"frame={r['frame_id']} "
        f"score={r['score']:.6f}"
    )

In [ ]:
fused = fusion_clip_caption(
    clip_results,
    caption_results
)

print("\n=== TOP 20 FUSION ===")

for i, r in enumerate(
    fused[:20],
    1
):
    print(
        f"{i:02d}. "
        f"video={r['video_id']} "
        f"frame={r['frame_id']} "
        f"score={r['score']:.6f} "
        f"clip={r['clip_score']:.6f} "
        f"caption={r['caption_score']:.6f}"
    )

In [ ]:
target = ("L21_V001", 12390)

print("\n=== CHECK FRAME 12390 ===")

for name, data in [
    ("CLIP", clip_results),
    ("CAPTION", caption_results),
    ("FUSION", fused)
]:

    found = False

    for rank, r in enumerate(data, 1):

        if (
            r["video_id"],
            r["frame_id"]
        ) == target:

            print(
                f"{name}: "
                f"rank={rank}, "
                f"score={r['score']:.6f}"
            )

            if name == "FUSION":
                print(
                    f"    clip_score="
                    f"{r['clip_score']:.6f}"
                )

                print(
                    f"    caption_score="
                    f"{r['caption_score']:.6f}"
                )

            found = True
            break

    if not found:
        print(f"{name}: NOT FOUND")

In [ ]:
from object_filter import ObjectMetadataLookup

object_lookup = ObjectMetadataLookup(
    data_dir="data"
)

print("Object Lookup OK")

In [ ]:
from apply_object_filter import apply_object_filter
filtered = apply_object_filter(
    fused[:20],
    structured_query["entities"],
    object_lookup
)

print("\n=== OBJECT FILTER ===")

for i, r in enumerate(filtered, 1):
    print(
        f"{i:02d}. "
        f"video={r['video_id']} "
        f"frame={r['frame_id']} "
        f"score={r['score']:.6f} "
        f"object={r.get('object_score', 0):.3f}"
    )

In [ ]:
from vlm_reranker import VLMReranker

vlm = VLMReranker()